<a href="https://colab.research.google.com/github/Olmioris/LLM-instruction-tuning-qwen2-qlora/blob/main/SFT_%D0%BD%D0%B0_Qwen2%E2%80%910_5B%E2%80%91Instruct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "transformers>=4.40.0" "datasets" "accelerate" "peft" "bitsandbytes" "trl>=1.9.2" "sentencepiece" "evaluate" "matplotlib" "pandas"
!pip install -q lm-eval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.2 MB/s eta 0:00:00


In [ ]:
import gc
import time
import torch, random, numpy as np
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from torch.profiler import profile, ProfilerActivity

SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)
print("CUDA available:", torch.cuda.is_available())

CUDA available: False


In [ ]:
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
MAX_SEQ_LENGTH = 512
DATASET_PATH = "/content/drive/MyDrive/MultiDomain_Instruction_50k"
OUTPUT_DIR = "/content/drive/MyDrive/Qwen2-0.5B-SFT-MultiDomain"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
raw_dataset = load_from_disk(DATASET_PATH)

def format_sft(example):
    instr = example["instruction"]
    inp = example["input"]
    out = example["output"]

    if inp and len(inp.strip()) > 0:
        user_text = instr + "\n\n" + inp
    else:
        user_text = instr

    return {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user",  "content": user_text},
            {"role": "assistant", "content": out},
        ]
    }

dataset = raw_dataset.map(format_sft)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def apply_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return example

dataset = dataset.map(apply_template)
dataset = dataset.train_test_split(test_size=0.02, seed=SEED)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="cpu",
)

baseline_tokenizer = tokenizer

def generate_baseline(prompt, max_new_tokens=200):
    inputs = baseline_tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = baseline_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return baseline_tokenizer.decode(output[0], skip_special_tokens=True)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
test_prompts = [
    "Объясни разницу между supervised и reinforcement learning.",
    "Придумай 3 идеи для улучшения мобильного банковского приложения.",
    "Сформулируй SQL-запрос для выборки клиентов с балансом выше 100000.",
    "Объясни, что такое A/B-тест и когда его стоит применять.",
    "Опиши, как продуктовый аналитик оценивает влияние новой фичи."
]

for i, p in enumerate(test_prompts, 1):
    print(f"\n=== PROMPT {i} ===\n{p}\n")
    print("BASELINE:\n", generate_baseline(p))


=== PROMPT 1 ===
Объясни разницу между supervised и reinforcement learning.

BASELINE:
 Объясни разницу между supervised и reinforcement learning. Судя по вашему запросу, вы хотите понять различия между supervised и reinforcement learning. В этом контексте, под "supervised" обычно означают, что обучение модели происходит с использованием предыдущих данных или знаний, которые она использует для обучения. Это включает в себя использование модели на основе существующего набора данных.

В то же время, "reinforcement learning" обычно означает процесс обучения модели, который требует выполнения действий, чтобы модель получила новые данные. Эти действия могут быть включать в себя нажатие кнопки, мониторинг состояния системы, или даже поведение, которое может зависеть от действий пользователя. 

Помимо этих двух терминов, есть еще множество других терминов, такие как "научная регрессия", "обучающая система", "интеллектуальная лог

=== PROMPT 2 ===
Придумай 3 идеи для улучшения мобильного банк

In [ ]:
!lm_eval \
  --model hf \
  --model_args pretrained={MODEL_NAME},dtype=float32 \
  --tasks hellaswag \
  --device cpu \
  --batch_size 1 \
  --limit 500 \
  --output_path /content/drive/MyDrive/lm_eval_baseline_qwen2_0.5b.json

2026-08-06:10:06:13 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-08-06:10:06:30 INFO     [_cli.run:388] Selected Tasks: ['hellaswag']
2026-08-06:10:06:30 WARNING  [evaluator:184] pretrained=Qwen/Qwen2-0.5B-Instruct appears to be an instruct or chat variant but chat template is not applied. Recommend setting
        `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-08-06:10:06:32 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-08-06:10:06:32 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2-0.5B-Instruct', 'dtype': 'float32'}
2026-08-06:10:06:38 INFO     [models.huggingface:286] Using device 'cpu'
2026-08-06:10:06:40 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': '

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cpu",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [ ]:
args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=20,
    max_steps=80,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=40,
    save_strategy="steps",
    save_steps=40,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_torch",
    fp16=False,
    bf16=False,
    report_to="none",
    seed=SEED,
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

batch = next(iter(trainer.get_train_dataloader()))

with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=False,
    profile_memory=False,
    with_stack=False
) as prof:
    outputs = model(**batch)
    loss = outputs.loss

print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=20))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                 Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                             aten::mm        68.08%        5.441s        68.09%        5.441s      18.828ms           289  
                        bitsandbytes::dequantize_4bit         8.83%     705.781ms         8.87%     709.056ms       4.221ms           168  
                                     aten::bernoulli_         4.09%     326.989ms         4.09%     326.989ms       3.406ms            96  
                                          aten::addmm         3.59%     287.206ms         3.66%     292.346ms       4.060ms            72  
                    

In [ ]:
train_out = trainer.train()
eval_out = trainer.evaluate()

print(f"Train loss: {train_out.training_loss:.4f}")
print(f"Eval loss: {eval_out['eval_loss']:.4f}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
40,1.517077,1.244427,1.254800,51473.000000,0.711041
80,1.132095,1.226462,1.226196,100837.000000,0.714435


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
1.132095,1.226463,80,1.226162,100837.000000,0.714435


Train loss: 1.3653
Eval loss: 1.2265


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
model.config.save_pretrained(OUTPUT_DIR)

print("Модель сохранена в:", OUTPUT_DIR)

Модель сохранена в: /content/drive/MyDrive/Qwen2-0.5B-SFT-MultiDomain
